<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code/06_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF   # 2-4에서 사용

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [6]:
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy(float)
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving knhanes_like_week4.csv to knhanes_like_week4.csv
Saving codebook_week4.csv to codebook_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


In [7]:
def _var_psu(z, strata, psu):
    """조사구 단위로 합산한 뒤 층 안에서 분산을 구한다 (궁극집락 방법)"""
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var


def svymean(y, w, strata, psu):
    """가중 평균/비율 + 설계 기반 표준오차 (4주차)"""
    y, w = np.asarray(y, float), np.asarray(w, float)
    ok = ~np.isnan(y)
    y, w = y[ok], w[ok]
    st, cl = np.asarray(strata)[ok], np.asarray(psu)[ok]
    th = np.sum(w * y) / np.sum(w)
    return th, np.sqrt(_var_psu(w * (y - th) / np.sum(w), st, cl))


print('함수 준비 완료')

함수 준비 완료


In [8]:
df['obes']    = (df['bmi'] >= 25).astype(float)
df['male']    = (df['sex'] == 1).astype(float)
df['smk_cur'] = (df['smk'] == 1).astype(float)

# 충돌부 — 비만과 고혈압이 모두 참여 확률을 높인다 (교육용, 씨앗 고정)
_rng = np.random.default_rng(20260922)
_p = 1 / (1 + np.exp(-(-1.6 + 1.5 * df['obes'] + 1.5 * df['htn'])))
df['prog'] = (_rng.random(len(df)) < _p).astype(float)

print('프로그램 참여율 (비가중)')
print(f"  비만 O · 고혈압 O : {df.loc[(df.obes==1)&(df.htn==1),'prog'].mean()*100:5.1f}%")
print(f"  비만 X · 고혈압 X : {df.loc[(df.obes==0)&(df.htn==0),'prog'].mean()*100:5.1f}%")

# 강의자가 제공하는 모집단 참값 (실제 연구에서는 알 수 없는 값)
TRUTH = {'M1': 1.891, 'M2': 1.687, 'M3': 1.005, 'M4': 1.016, 'M5': 1.664}
print()
print('모집단 참값(목표 추정량) = OR', TRUTH['M2'])

프로그램 참여율 (비가중)
  비만 O · 고혈압 O :  78.7%
  비만 X · 고혈압 X :  16.8%

모집단 참값(목표 추정량) = OR 1.687


In [10]:
# TODO: 점수(score)의 궁극집락 분산행렬 B를 계산하는 함수를 완성하십시오.
#       힌트: _var_psu 와 구조가 같습니다. 스칼라 → 행렬로만 바뀝니다.

def _vcl(U, strata, psu):
    """점수행렬 U(n×p)를 조사구별로 합산한 뒤 층 안에서 분산행렬을 구한다"""
    t = pd.DataFrame(U); t['st'] = strata; t['cl'] = psu
    uc = t.groupby(['st', 'cl']).sum(numeric_only=True)   # 조사구별 점수합
    p = U.shape[1]
    B = np.zeros((p, p))
    for s, g in uc.groupby(level=0):
        m = len(g)                      # 층 안 조사구 수
        if m > 1:
            D = g.values - g.values.mean(axis=0)    # TODO: 조사구 점수합 − 층 평균
            B += m/(m-1) * D.T @ D             # TODO: m/(m-1) * D'D
    return B


def svyglm(y, X, w, strata, psu, family='binomial'):
    y = np.asarray(y, float); X = np.asarray(X, float); w = np.asarray(w, float)
    if family == 'binomial':
        m = sm.GLM(y, X, family=sm.families.Binomial(),
                   freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = mu * (1 - mu)              # 이분형의 분산함수
    else:
        m = sm.GLM(y, X, family=sm.families.Gaussian(),
                   freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = np.ones_like(mu)           # 정규의 분산함수
    b = m.params
    A = (X * (w*v) [:, None]).T @ X         # TODO: X' diag(w*v) X
    U = (w*(y - mu))[:, None] * X         # TODO: 점수 w*(y-mu)*x  → n×p 행렬
    B = _vcl(U, strata, psu)
    Ai = np.linalg.inv(A)
    V  = Ai @ B @ Ai                    # 샌드위치
    return b, np.sqrt(np.diag(V)), m


# 검증 ① — 절편만 있는 로지스틱의 계수는 가중유병률의 로짓과 같다
X0 = np.ones((len(df), 1))
b0, se0, _ = svyglm(df['htn'], X0, w, ST, PS)
p_w = np.average(df['htn'], weights=w)
print(f"절편     = {b0[0]:.6f}   logit(가중유병률) = {np.log(p_w/(1-p_w)):.6f}")

# 검증 ② — 절편의 설계 기반 SE는 4주차 svymean의 SE를 델타법으로 바꾼 값과 같다
#   계수는 statsmodels가 계산하므로 ①만으로는 여러분이 채운 A·U·B를 검증할 수 없습니다.
#   ②가 맞아야 _vcl과 svyglm이 맞은 것입니다.
m_w, se_w = svymean(df['htn'], w, ST, PS)
print(f"SE(절편) = {se0[0]:.6f}   델타법 SE        = {se_w/(p_w*(1-p_w)):.6f}")
assert np.isclose(se0[0], se_w / (p_w * (1 - p_w))), 'SE가 다릅니다 — _vcl, A, U를 다시 확인하십시오'
print()
print('✅ svyglm 준비 완료 — 두 줄의 값이 같으면 다음으로 넘어가십시오')

절편     = -1.132045   logit(가중유병률) = -1.132045
SE(절편) = 0.041826   델타법 SE        = 0.041826

✅ svyglm 준비 완료 — 두 줄의 값이 같으면 다음으로 넘어가십시오


In [12]:
# TODO: M1(무보정)과 M2(보정집합)를 적합하고 OR과 95% CI를 출력하십시오.
#       참값과 나란히 놓고 신뢰구간이 참값을 포함하는지 확인하십시오.

def or_ci(b, se, i=1):
    """i번째 계수의 OR과 95% CI"""
    return np.exp(b[i]), np.exp(b[i] - 1.96*se[i]), np.exp(b[i] + 1.96*se[i])        # TODO: exp(b), exp(b-1.96se), exp(b+1.96se)


EST = {}                           # 모형별 결과 저장소 — 1-4 ~ 1-6, 2-2, 2-5에서 다시 씁니다

for key, cols in [('M1', ['obes']),
                  ('M2', ['obes', 'age', 'male', 'smk_cur'])]:
    X = np.column_stack([np.ones(len(df)), df[cols].values])     # TODO: 절편 추가한 설계행렬
    b, se, m =  svyglm(df['htn'], X, w, ST, PS)     # TODO: svyglm 호출
    o, lo, hi = or_ci(b, se)
    EST[key] = {'cols': cols, 'b': b, 'se': se, 'model': m, 'or': (o, lo, hi)}
    print(f"{key}  OR = {o:.2f} ({lo:.2f}-{hi:.2f})  참값 1.69  포함: {lo <= 1.69 <= hi}")    # TODO: 모형 이름, OR (95% CI), 참값, 참값 포함 여부

M1  OR = 1.91 (1.68-2.17)  참값 1.69  포함: True
M2  OR = 1.69 (1.44-1.97)  참값 1.69  포함: True


In [13]:
# B-2: 공복혈당 ≥100 → 고혈압
df['hglu']       = (df['glucose'] >= 100).astype(float)          # 결측은 False가 되지만 아래에서 제외
df['smk_former'] = (df['smk'] == 2).astype(float)
df['drink_mod']  = (df['drink'] == 2).astype(float)
df['drink_high'] = (df['drink'] == 3).astype(float)
df['exer']       = (df['exercise'] == 2).astype(float)

# 결측: glucose 공백(검사 미시행), edu/income 9(무응답) → 가중치 0 (도메인 방식, 설계 정보 유지)
ok = (df['glucose'].notna() & (df['edu'] != 9) & (df['income'] != 9)).values
wd = w * ok
print(f"분석 {ok.sum():,}명 / 제외 {(~ok).sum()}명 (glucose 결측 {df.glucose.isna().sum()}, edu=9 {(df.edu==9).sum()}, income=9 {(df.income==9).sum()})")

ADJ = ['age', 'male', 'bmi', 'smk_cur', 'smk_former', 'drink_mod', 'drink_high', 'exer', 'edu', 'income']
EST2 = {}
for key, cols in [('M1 무보정', ['hglu']), ('M2 보정', ['hglu'] + ADJ)]:
    X = np.column_stack([np.ones(len(df)), df[cols].values])
    b, se, m = svyglm(df['htn'], X, wd, ST, PS)
    o, lo, hi = or_ci(b, se)
    EST2[key] = {'cols': cols, 'b': b, 'se': se, 'or': (o, lo, hi)}
    print(f"{key}  OR = {o:.2f} ({lo:.2f}-{hi:.2f})")

분석 4,635명 / 제외 732명 (glucose 결측 367, edu=9 80, income=9 326)
M1 무보정  OR = 1.68 (1.45-1.95)
M2 보정  OR = 1.01 (0.85-1.19)
